In [ ]:
import pandas as pd
import numpy as np  
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score 
from sklearn.utils import resample
import matplotlib.pyplot as plt
import os
import joblib  # for model saving 

# ==========================================
# 0. Data Loading and Preprocessing 
# ==========================================

# Define file path 
file_path = "example_data.csv"

# Read dataset 
data = pd.read_csv(file_path)  

# Specify categorical variables 
categorical_cols = ["N", "O"]
for col in categorical_cols:
    data[col] = data[col].astype('category')

# Define features (X) and target (y) 
features = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N", "O"]
X = data[features]
y = data['Target']

# Randomly partition data (85% Train, 15% Test) 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# ==========================================
# 1. Stage 1: Grid Search & 10-fold CV 
# ==========================================
print("Starting Grid Search and 10-fold CV... ")

# Initialize LightGBM regressor 
lgb_model = lgb.LGBMRegressor(objective='regression', random_state=42, n_jobs=-1)

# Define hyperparameter grid 
param_grid = {
    'learning_rate': [0.01, 0.05],
    'max_depth': [5, 7],
    'num_leaves': [18, 31],
    'n_estimators': [500, 1000], 
    'subsample': [0.8],
    'colsample_bytree': [0.7]
}

# Configure Grid Search with 10-fold CV 
grid_search = GridSearchCV(
    estimator=lgb_model, 
    param_grid=param_grid, 
    cv=10, 
    scoring='neg_root_mean_squared_error', 
    verbose=1,
    n_jobs=-1
)

# Fit on training set only to prevent data leakage 
grid_search.fit(X_train, y_train)

print(f"Best hyperparameters: {grid_search.best_params_} ")
best_model = grid_search.best_estimator_

# Evaluate on overall training set
y_pred_train = best_model.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
mae_train = mean_absolute_error(y_train, y_pred_train) 
r2_train = r2_score(y_train, y_pred_train)

# ==========================================
# 2. Stage 2: Bootstrap Resampling on Test Set 
# ==========================================
print("\nStarting Bootstrap resampling (100 iterations)... ")

n_iterations = 100
bootstrap_rmse = []
bootstrap_mae = [] 
bootstrap_r2 = []

for i in range(n_iterations):
    # Resample test set with replacement 
    X_boot, y_boot = resample(X_test, y_test, replace=True, random_state=i)
    
    # Predict on resampled test set 
    y_pred_boot = best_model.predict(X_boot)
    
    # Recalculate evaluation metrics 
    rmse_b = np.sqrt(mean_squared_error(y_boot, y_pred_boot))
    mae_b = mean_absolute_error(y_boot, y_pred_boot) 
    r2_b = r2_score(y_boot, y_pred_boot)
    
    bootstrap_rmse.append(rmse_b)
    bootstrap_mae.append(mae_b)
    bootstrap_r2.append(r2_b)

# Calculate mean and standard deviation 
rmse_mean, rmse_std = np.mean(bootstrap_rmse), np.std(bootstrap_rmse)
mae_mean, mae_std = np.mean(bootstrap_mae), np.std(bootstrap_mae)
r2_mean, r2_std = np.mean(bootstrap_r2), np.std(bootstrap_r2)

print("\n=== Final Evaluation Results ")
print(f"Training Set - RMSE: {rmse_train:.4f}, MAE: {mae_train:.4f}, R2: {r2_train:.4f}")
print(f"Test Set (Bootstrap Mean ± 1 SD) - RMSE: {rmse_mean:.4f} ± {rmse_std:.4f}, MAE: {mae_mean:.4f} ± {mae_std:.4f}, R2: {r2_mean:.4f} ± {r2_std:.4f}")

# Calculate overall test set predictions)
y_pred_test_overall = best_model.predict(X_test)

# ==========================================
# 3. Data Integration and Export 
# ==========================================
print("\nExporting predicted results... ")

data['Predicted_Value'] = np.nan
data['Dataset_Type'] = np.nan 

# Assign predictions to corresponding indices 
data.loc[y_train.index, 'Predicted_Value'] = y_pred_train
data.loc[y_train.index, 'Dataset_Type'] = 'Train'
data.loc[y_test.index, 'Predicted_Value'] = y_pred_test_overall
data.loc[y_test.index, 'Dataset_Type'] = 'Test'

output_filename = "Full_Data_With_Predictions.csv"
try:
    data.to_csv(output_filename, index=False)
    print(f"Export successful: {output_filename} ")
except Exception as e:
    print(f"Export failed : {e}")

# ==========================================
# 4. Visualization (Scatter Plots) )
# ==========================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot Training Set
ax1.scatter(y_train, y_pred_train, alpha=0.5, color='blue')
lims_train = [np.min([ax1.get_xlim(), ax1.get_ylim()]), np.max([ax1.get_xlim(), ax1.get_ylim()])]
ax1.plot(lims_train, lims_train, 'r--', lw=2)
ax1.set_xlabel('Actual Values')
ax1.set_ylabel('Predicted Values')
ax1.set_title(f'Training Set\nR² = {r2_train:.4f}, RMSE = {rmse_train:.4f}, MAE = {mae_train:.4f}')
ax1.grid(True, linestyle='--', alpha=0.7)

# Plot Test Set 
overall_rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test_overall))
overall_mae_test = mean_absolute_error(y_test, y_pred_test_overall)
overall_r2_test = r2_score(y_test, y_pred_test_overall)

ax2.scatter(y_test, y_pred_test_overall, alpha=0.5, color='green')
lims_test = [np.min([ax2.get_xlim(), ax2.get_ylim()]), np.max([ax2.get_xlim(), ax2.get_ylim()])]
ax2.plot(lims_test, lims_test, 'r--', lw=2)
ax2.set_xlabel('Actual Values')
ax2.set_ylabel('Predicted Values')
ax2.set_title(f'Test Set (Overall)\nR² = {overall_r2_test:.4f}, RMSE = {overall_rmse_test:.4f}, MAE = {overall_mae_test:.4f}')
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()

# ==========================================
# 5. Save Figures 
# ==========================================
plot_png = "Model_Performance_Scatter.png"
plot_pdf = "Model_Performance_Scatter.pdf" 
try:
    plt.savefig(plot_png, dpi=300, bbox_inches='tight')
    plt.savefig(plot_pdf, dpi=300, bbox_inches='tight')
    print(f"Figures saved successfully : {plot_png}, {plot_pdf}")
except Exception as e:
    print(f"Failed to save figures : {e}")

# ==========================================
# 6. Save the Trained Model 
# ==========================================
print("\nSaving the trained model... ")
model_filename = "best_lightgbm_model.pkl"
try:
    joblib.dump(best_model, model_filename)
    print(f"Model saved successfully: {model_filename} ")
except Exception as e:
    print(f"Failed to save the model : {e}")

plt.show()